# Harbor House Café — written sentiment

## Objectives

WeLoveReviews is about to hand a client 500 written reviews. The business average on those reviews is 4.5 stars out of 5. The account manager wants a second read of the text itself: how many reviews read as positive, neutral, or negative, and whether that split still supports a 4.5 average. If it does not, the gap has to be visible before the report goes to the client.


## Explore the file

The table is `data/raw/reviews.csv`. The next cell checks the shape, the column types, missing values, the star distribution, how long the text is, and a few raw reviews.


In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "raw" / "reviews.csv").exists():
    ROOT = ROOT.parent

reviews = pd.read_csv(ROOT / "data" / "raw" / "reviews.csv")
reviews["rating"] = reviews["rating"].astype(int)
reviews["text_characters"] = reviews["review_text"].str.len()
reviews["text_words"] = reviews["review_text"].str.split().str.len()

print("shape", reviews.shape)
print(reviews.dtypes.to_string())
print("missing values")
print(reviews[["review_id", "rating", "review_text"]].isna().sum().to_string())
print("blank text", int((reviews["review_text"].str.strip() == "").sum()))
print("duplicate review_id", int(reviews["review_id"].duplicated().sum()))
print("duplicate review text", int(reviews["review_text"].duplicated().sum()))
print("rating counts")
print(reviews["rating"].value_counts().sort_index().to_string())
print("mean rating", round(reviews["rating"].mean(), 2))
print(
    "text characters min/mean/max",
    int(reviews["text_characters"].min()),
    round(reviews["text_characters"].mean(), 1),
    int(reviews["text_characters"].max()),
)
print(
    "text words min/mean/max",
    int(reviews["text_words"].min()),
    round(reviews["text_words"].mean(), 1),
    int(reviews["text_words"].max()),
)
reviews[["review_id", "rating", "review_text"]].head(5)


shape (500, 5)
review_id          int64
rating             int64
review_text          str
text_characters    int64
text_words         int64
missing values
review_id      0
rating         0
review_text    0
blank text 0
duplicate review_id 0
duplicate review text 7
rating counts
rating
1     12
2     18
3     38
4     72
5    360
mean rating 4.5
text characters min/mean/max 86 170.2 230
text words min/mean/max 14 29.1 42


,review_id,rating,review_text
0,1,5,Visited Harbor House Café last weekend. Great ...
1,2,5,Tried Harbor House Café after seeing it recomm...
2,3,5,Tried Harbor House Café after seeing it recomm...
3,4,5,Visited Harbor House Café last weekend. The st...
4,5,5,Grabbed a quick coffee at Harbor House Café th...


## Insights

Four facts from that table decide what to do next.

- The 4.5 average is the mean of these 500 stars, and it is exact. It is not a small group of perfect scores hiding a large group of poor ones: 360 reviews are 5 stars, 72 are 4 stars, and only 30 are 1 or 2 stars.
- Using the same bands the model will use, the human stars are 86.4% positive (4–5), 7.6% neutral (3), and 6.0% negative (1–2).
- Every row has an id, a star from 1 to 5, and text. The text is short, 14 to 42 words, so a transformer can read the whole review.
- Seven review texts appear twice (14 rows). Each pair carries the same star. They are repeated submissions, not a reason to shrink the file below the 500 reviews the client is waiting on.


## Cleaning

No rows should be dropped or rewritten before scoring. There is nothing missing to fill, no star outside 1–5, and no empty review. The seven repeated texts stay, because each one has its own `review_id` and the count has to remain 500. The check below is the whole cleaning decision.


In [2]:
invalid_stars = ~reviews["rating"].between(1, 5)
blank_text = reviews["review_text"].str.strip().eq("")
print("rows", len(reviews))
print("invalid stars", int(invalid_stars.sum()))
print("blank text", int(blank_text.sum()))
print("rows removed", 0)


rows 500
invalid stars 0
blank text 0
rows removed 0


## Action plan and model

The star column already averages 4.5. Another average of that column would not answer the account manager. The open question is whether the sentences read the same way.

The model for that read is fixed: `nlptown/bert-base-multilingual-uncased-sentiment`, revision `8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d`. It returns a star from 1 to 5, not a positive/negative tag. Those stars map as 1–2 negative, 3 neutral, and 4–5 positive.

This checkpoint was fine-tuned on product reviews. Harbor House Café reviews talk about staff, wait time, and the room. That mismatch can mark a warm service review as a low star. The model stays the one above anyway, so those misses can be found and explained. It is loaded once, through `pipeline()`, at that revision. The weights are not stored in the repository.


## Score all 500 reviews

The next cell loads that checkpoint a single time and stores a predicted star and a sentiment band on every row.


In [3]:
import sys

sys.path.insert(0, str(ROOT / "src"))

from app import MODEL_ID, MODEL_REVISION, load_classifier, score_frame

print(MODEL_ID)
print(MODEL_REVISION)
classifier = load_classifier()
scored = score_frame(reviews.drop(columns=["text_characters", "text_words"]), classifier)
print("rows scored", len(scored))
print("missing predictions", int(scored["predicted_stars"].isna().sum()))
scored[["review_id", "rating", "predicted_stars", "sentiment_band", "model_score"]].head(8)


nlptown/bert-base-multilingual-uncased-sentiment
8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d


rows scored 500
missing predictions 0


,review_id,rating,predicted_stars,sentiment_band,model_score
0,1,5,5,positive,0.7170
1,2,5,5,positive,0.9822
2,3,5,5,positive,0.6623
3,4,5,5,positive,0.9895
4,5,5,5,positive,0.9829
5,6,5,5,positive,0.3339
6,7,5,5,positive,0.7980
7,8,5,5,positive,0.8237


## Breakdown against the 4.5-star average

The next cell puts the human stars and the model stars on the same bands and compares both with the 4.5 average.


In [4]:
def band(stars):
    if stars <= 2:
        return "negative"
    if stars == 3:
        return "neutral"
    return "positive"

human_band = scored["rating"].map(band)
model_band = scored["sentiment_band"]
summary = pd.DataFrame(
    {
        "human_reviews": human_band.value_counts(),
        "human_percent": (human_band.value_counts(normalize=True) * 100).round(1),
        "model_reviews": model_band.value_counts(),
        "model_percent": (model_band.value_counts(normalize=True) * 100).round(1),
    }
).reindex(["positive", "neutral", "negative"])
print("human mean stars", round(scored["rating"].mean(), 2))
print("model mean stars", round(scored["predicted_stars"].mean(), 2))
print("same star", int((scored["rating"] == scored["predicted_stars"]).sum()), "of", len(scored))
print("same band", int((human_band == model_band).sum()), "of", len(scored))
summary


human mean stars 4.5
model mean stars 4.4
same star 429 of 500
same band 475 of 500


,human_reviews,human_percent,model_reviews,model_percent
positive,432,86.4,412,82.4
neutral,38,7.6,41,8.2
negative,30,6.0,47,9.4


The written file still reads as a high-scoring café. The model calls 82.4% of reviews positive and 9.4% negative, against 86.4% and 6.0% on the human stars. Its mean is 4.40 stars, against the business average of 4.50. The same band matches on 475 of 500 reviews. The gap is real, and it is small: the model is a little harsher, not a different story. The next section is where those extra negative calls come from.


## False negatives

A false negative here is a review the model scores 1 or 2 while the customer left 4 or 5, or a review that reads as positive or neutral while the model disagrees. The cell lists every 4–5 star review the model placed in the negative band.


In [5]:
false_negatives = scored[(scored["predicted_stars"] <= 2) & (scored["rating"] >= 4)].sort_values(
    ["predicted_stars", "model_score", "review_id"]
)
print("false negatives", len(false_negatives), "of", int((scored["rating"] >= 4).sum()), "high-star reviews")
print("mean model score on those rows", round(false_negatives["model_score"].mean(), 3))
print("mean model score on all rows", round(scored["model_score"].mean(), 3))
false_negatives[["review_id", "rating", "predicted_stars", "model_score", "review_text"]]


false negatives 16 of 432 high-star reviews
mean model score on those rows 0.378
mean model score on all rows 0.704


,review_id,rating,predicted_stars,model_score,review_text
72,73,5,1,0.2679,Stopped by Harbor House Café for the first tim...
331,332,5,1,0.3144,Stopped by Harbor House Café for the first tim...
86,87,5,1,0.3247,Tried Harbor House Café after seeing it recomm...
488,489,5,1,0.3331,Tried Harbor House Café after seeing it recomm...
203,204,5,1,0.3492,Came to Harbor House Café for a birthday brunc...
253,254,5,1,0.3641,Stopped by Harbor House Café for the first tim...
14,15,5,1,0.3893,Every dish was bursting with flavor. The place...
269,270,5,1,0.4091,Stopped by Harbor House Café for the first tim...
408,409,5,1,0.4391,Tried Harbor House Café after seeing it recomm...
13,14,4,1,0.4443,Tried Harbor House Café after seeing it recomm...


Sixteen reviews fall in that set. Fourteen of them have no complaint in the text at all. Review 15 is five stars and only says the dishes were full of flavor, the room felt welcoming, and the guest will be back. The model calls it 1 star with a score of 0.39. Reviews 73, 87, 14, 270, and 489 are the same kind of praise: fast service, an allergy handled with care, a server who remembered names, a team that cares. Their top scores sit between 0.27 and 0.48, so the 1-star label is a weak win, not a confident reading.

Two of the sixteen do contain a complaint, and the model lets that clause take the whole review. Review 322 says the staff seemed annoyed and then says the food made up for it and the guest will be back. Review 204 says the cashier was rude and then says the same thing. Both are five-star reviews. The model scores them 1 star. Product-review training is sharp about defect words such as "rude" and "annoyed", and it is weak on café praise about the room, a menu, or a member of staff. The low scores on the clean praise, and the overweight complaint in the two mixed reviews, are the shared pattern.

The sentence "Every dish was bursting with flavor" shows up in 12 of the 16, but it also shows up in four- and five-star reviews the model scored as positive, including review 1. The sentence is not itself a negative signal.


## Hand check of 18 reviews

These 18 rows were read one by one. The note says whether the model band matches the text.


In [6]:
hand_ids = [15, 73, 87, 14, 322, 204, 32, 100, 1, 2, 9, 16, 192, 230, 17, 19, 265, 77]
hand = scored.set_index("review_id").loc[hand_ids].reset_index()
hand[["review_id", "rating", "predicted_stars", "sentiment_band", "model_score", "review_text"]]


,review_id,rating,predicted_stars,sentiment_band,model_score,review_text
0,15,5,1,negative,0.3893,Every dish was bursting with flavor. The place...
1,73,5,1,negative,0.2679,Stopped by Harbor House Café for the first tim...
2,87,5,1,negative,0.3247,Tried Harbor House Café after seeing it recomm...
3,14,4,1,negative,0.4443,Tried Harbor House Café after seeing it recomm...
4,322,5,1,negative,0.6077,Every dish was bursting with flavor. The staff...
5,204,5,1,negative,0.3492,Came to Harbor House Café for a birthday brunc...
6,32,2,1,negative,0.5965,Stopped by Harbor House Café for the first tim...
7,100,1,2,negative,0.4336,Came to Harbor House Café for a birthday brunc...
8,1,5,5,positive,0.7170,Visited Harbor House Café last weekend. Great ...
9,2,5,5,positive,0.9822,Tried Harbor House Café after seeing it recomm...


| review_id | Human star | Model star | Does the label fit the text? |
| --- | --- | --- | --- |
| 15 | 5 | 1 | No. The text is only praise. A 1-star label does not fit. The score is 0.39, so the model is unsure. |
| 73 | 5 | 1 | No. Fast, friendly service and food the guest will pay for again. The label is too low. |
| 87 | 5 | 1 | No. The seasonal menu and the allergy request are both told as good news. Nothing in the text is negative. |
| 14 | 4 | 1 | No. Flavor and a team that cares, and the guest will return. The human 4 and the model 1 are not the same read. |
| 322 | 5 | 1 | No. "The staff seemed annoyed" is a real complaint, and the next sentence takes it back. A person still hears a five-star recovery. The model stops on the complaint. This is the most confident miss, at 0.61. |
| 204 | 5 | 1 | No. Same shape as 322: a rude cashier, then "the food made up for it." The model treats the defect as the whole review. |
| 32 | 2 | 1 | Yes. Cold eggs, a rude cashier, and no plan to return. Negative is the right band. |
| 100 | 1 | 2 | Yes. Burnt coffee, a rude cashier, and a cramped room. The band is negative on both sides. |
| 1 | 5 | 5 | Yes. A good place to sit, good food, and worth the price. |
| 2 | 5 | 5 | Yes. The coffee, the value, and "five stars without hesitation" all point the same way. Score 0.98. |
| 9 | 3 | 3 | Yes. Average sandwiches and polite, slow staff. "Fine for a quick stop" is a neutral review. |
| 16 | 3 | 3 | Yes. A repetitive menu and a rude cashier, with a willingness to try again. Neutral fits better than a 1. |
| 192 | 2 | 3 | The model is closer to the text. "Decent", "fine", and "nothing memorable" are lukewarm, not a failed meal. The stored 2 stars are harsher than the sentences. |
| 230 | 2 | 3 | Same reading as 192. Polite but slow, and not loved enough to repeat, is neutral language. The model did not call it positive. |
| 17 | 4 | 5 | The band is right either way. The text itself says "five stars without hesitation" after noting slow staff. The model followed that sentence. The stored 4 is the softer number. |
| 19 | 4 | 4 | Yes. Strong pastries, slow but polite staff, and "might give it another shot." A 4 is a fair read. |
| 265 | 5 | 5 | Yes. "Best brunch" and "five stars without hesitation," with almost no doubt in the score. |
| 77 | 5 | 4 | The band is right. A rude cashier is in the text, and so is the recovery. Dropping from 5 to 4 is a small move, unlike the jump to 1 star on reviews 322 and 204. The score is only 0.50. |


## Saved file

`src/app.py` is the production path. It performs this same load once and writes `data/processed/reviews_with_sentiment.csv`. The columns are the original review plus `predicted_stars`, `sentiment_band`, and `model_score`.


In [7]:
saved = pd.read_csv(ROOT / "data" / "processed" / "reviews_with_sentiment.csv")
print(saved.shape)
print(saved.columns.tolist())
print("rows with a band", int(saved["sentiment_band"].notna().sum()))
print(
    "same predictions as this notebook",
    bool((saved.sort_values("review_id")["predicted_stars"].to_numpy() == scored.sort_values("review_id")["predicted_stars"].to_numpy()).all()),
)


(500, 6)
['review_id', 'rating', 'review_text', 'predicted_stars', 'sentiment_band', 'model_score']
rows with a band 500
same predictions as this notebook True


## Conclusions

The written reviews support the 4.5-star average. The stars in the file average 4.50, and 86.4% of them are 4 or 5. A product-review model, which had not seen this café, still calls 82.4% of the same texts positive and averages 4.40 stars. Same-band agreement is 475 of 500.

The gap is 16 reviews that customers scored 4 or 5 and the model scored 1 or 2. Fourteen of those texts contain no complaint. The model is unsure on them. The other two mention a rude cashier or annoyed staff and then take it back, and the model keeps only the complaint. No 1-star or 2-star review was called positive. The average is not hiding a large group of unhappy guests.

For the client, the second read can say the writing matches the 4.5 average. The 16 misses belong in the appendix, with the note that they come from a product-review model reading service praise, not from a different rating of the café.


## Addendum: a second model

The required score above stays on `nlptown/bert-base-multilingual-uncased-sentiment`. This addendum runs `tabularisai/multilingual-sentiment-analysis` at revision `eea032081f8d247b4303ef3565e7cec1b6f201c9` on the same 500 reviews. Its labels are Very Negative, Negative, Neutral, Positive, and Very Positive. Very Negative and Negative are the negative band, the counterpart of a 1 or 2 star score. A false negative is still a 4 or 5 star review placed in that negative band.


In [8]:
from transformers import pipeline

ALT_MODEL_ID = "tabularisai/multilingual-sentiment-analysis"
ALT_REVISION = "eea032081f8d247b4303ef3565e7cec1b6f201c9"
alt_classifier = pipeline(
    "sentiment-analysis",
    model=ALT_MODEL_ID,
    revision=ALT_REVISION,
    truncation=True,
    device=-1,
)
alt_predictions = alt_classifier(scored["review_text"].astype(str).tolist(), batch_size=32)
alt = scored[["review_id", "rating", "review_text", "predicted_stars"]].copy()
alt["alt_label"] = [item["label"] for item in alt_predictions]
alt["alt_score"] = [round(float(item["score"]), 4) for item in alt_predictions]
alt["alt_band"] = alt["alt_label"].map(
    lambda label: "negative" if label in {"Very Negative", "Negative"} else "neutral" if label == "Neutral" else "positive"
)
high = alt["rating"] >= 4
nlptown_fn = alt[high & (alt["predicted_stars"] <= 2)]
alt_fn = alt[high & (alt["alt_band"] == "negative")]
print(ALT_MODEL_ID)
print(ALT_REVISION)
print("nlptown false negatives", len(nlptown_fn), "of", int(high.sum()))
print("second model false negatives", len(alt_fn), "of", int(high.sum()))
print("still missed by both", sorted(set(nlptown_fn["review_id"]) & set(alt_fn["review_id"])))
print("fixed by the second model", sorted(set(nlptown_fn["review_id"]) - set(alt_fn["review_id"])))
print("new misses", sorted(set(alt_fn["review_id"]) - set(nlptown_fn["review_id"])))
alt_fn[["review_id", "rating", "predicted_stars", "alt_label", "alt_score", "review_text"]]


tabularisai/multilingual-sentiment-analysis
eea032081f8d247b4303ef3565e7cec1b6f201c9
nlptown false negatives 16 of 432
second model false negatives 7 of 432
still missed by both [254]
fixed by the second model [14, 15, 73, 87, 138, 200, 204, 270, 322, 332, 409, 440, 462, 474, 489]
new misses [139, 220, 231, 290, 352, 415]


,review_id,rating,predicted_stars,alt_label,alt_score,review_text
138,139,5,5,Negative,0.4994,Tried Harbor House Café after seeing it recomm...
219,220,5,5,Negative,0.4731,Visited Harbor House Café last weekend. They w...
230,231,5,5,Negative,0.4662,They went out of their way to accommodate my a...
253,254,5,1,Negative,0.6090,Stopped by Harbor House Café for the first tim...
289,290,5,5,Negative,0.4001,Grabbed a quick coffee at Harbor House Café th...
351,352,5,5,Very Negative,0.4740,They went out of their way to accommodate my a...
414,415,5,5,Negative,0.3796,Came to Harbor House Café for a birthday brunc...


The false-negative count falls from 16 to 7, which is 3.7% of the 432 high-star reviews down to 1.6%. Fifteen of the original misses are recovered, including reviews 204 and 322, where a rude cashier or annoyed staff is followed by a sentence that takes the complaint back. Review 254 is the only review both models call negative. It is five stars and contains no complaint.

The seven reviews the second model still calls negative are 139, 220, 231, 254, 290, 352, and 415. Each one includes "They went out of their way to accommodate my allergy." Thirty-one other high-star reviews use that same sentence and are scored positive, so the sentence is not a switch that always flips the label. The scores on these seven are mostly weak, from 0.38 to 0.61. Review 254, at 0.61, is the most confident of them.

The second model is looser on lukewarm writing. It calls 24 of the 38 three-star reviews positive. Review 192 is a stored 2: the food was decent, the service was forgettable, and the guest did not want to come back. The second model calls that review positive.

WeLoveReviews should use the second model, not the product-review model, as the client-facing second read. It cuts the false negatives by more than half and it understands the mixed reviews the first model flattened into a complaint. It should not be the only pass. The seven allergy-line misses, and the lukewarm reviews it marks positive, still need the same kind of hand check used above. The script in `src/app.py` stays on the required model and revision from the main analysis.
